# ZG549 Assignment II - Section II: Fine-Tuning
**Group 32 | Sentiment model for the Equity Research Desk Assistant**

Fine-tunes `distilbert-base-uncased` on the **Financial PhraseBank** dataset
(`sentences_allagree`: ~2.3k financial sentences where all annotators agreed on
the positive / neutral / negative label), and compares against pretrained
**FinBERT** as the baseline.

Run on Google Colab (Runtime -> Change runtime type -> **T4 GPU**). End-to-end ~10 min.
The final cell zips the fine-tuned model; download it and unzip into the app's
`models/finetuned-sentiment/` folder.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn evaluate

In [ ]:
from datasets import load_dataset

LABELS = ["negative", "neutral", "positive"]  # dataset label order: 0,1,2
ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
ds = ds["train"].train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
train_ds, test_ds = ds["train"], ds["test"]
print(train_ds, test_ds)
print("Label distribution (train):", sorted(
    __import__("collections").Counter(train_ds["label"]).items()))

## Baseline: pretrained FinBERT on our test split
FinBERT is already finance-tuned, so it is a strong baseline; the point of this
section is to show the fine-tuning *process* and that a small generic model
(DistilBERT) reaches comparable quality after task-specific fine-tuning.

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

finbert = pipeline("text-classification", model="ProsusAI/finbert", device=0)
pred = finbert(test_ds["sentence"], truncation=True, batch_size=64)
finbert_pred = [LABELS.index(p["label"].lower()) for p in pred]
finbert_acc = accuracy_score(test_ds["label"], finbert_pred)
print(f"FinBERT baseline accuracy: {finbert_acc:.4f}")
print(classification_report(test_ds["label"], finbert_pred, target_names=LABELS))

## Fine-tune DistilBERT

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import numpy as np, evaluate

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["sentence"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3,
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id={l: i for i, l in enumerate(LABELS)},
)

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, -1), references=labels)

args = TrainingArguments(
    output_dir="ft-out",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok,
                  eval_dataset=test_tok, tokenizer=tok,
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)
trainer.train()

## Evaluate the fine-tuned model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

logits = trainer.predict(test_tok).predictions
ft_pred = logits.argmax(-1)
ft_acc = accuracy_score(test_ds["label"], ft_pred)

print(f"FinBERT baseline accuracy   : {finbert_acc:.4f}")
print(f"Fine-tuned DistilBERT acc   : {ft_acc:.4f}")
print()
print(classification_report(test_ds["label"], ft_pred, target_names=LABELS))

cm = confusion_matrix(test_ds["label"], ft_pred)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues")
plt.title(f"Fine-tuned DistilBERT - Financial PhraseBank (acc {ft_acc:.3f})")
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

## Save and download
Download `finetuned-sentiment.zip`, unzip into the app repo at
`models/finetuned-sentiment/`, and the Sentiment tab will offer the
fine-tuned model alongside FinBERT.

In [ ]:
trainer.save_model("finetuned-sentiment")
tok.save_pretrained("finetuned-sentiment")
!zip -rq finetuned-sentiment.zip finetuned-sentiment confusion_matrix.png
from google.colab import files
files.download("finetuned-sentiment.zip")